In [ ]:
import torch
import numpy as np
import torch.nn.functional as F
from datetime import datetime
from pathlib import Path
import sys


PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

device = torch.device("cuda" if torch.cuda.is_available()
                else ("mps" if torch.backends.mps.is_available()
                else "cpu"))
print(F"Device set to {device}")


In [ ]:
from Datasets.DataLoader import CombinedBinDataLoader
from Models.Lag_Behind_Model.Lag_Behind_Model import GPT2_Lag
from Models.Configs import LagBehindConfig, TrainConfig
from Models.Lag_Behind_Model.train_Lag_Behind import train_loop, estimate_loss
from Models.Lag_Behind_Model.test_Lag_Behind import evaluate_next_token_correction

In [ ]:
model_config = LagBehindConfig()
train_config = TrainConfig()
eff_batch_size = train_config.batch_per_iter * train_config.grad_acc_factor
tokens_per_step = eff_batch_size * model_config.block_size


model = GPT2_Lag(model_config, device)
model = model.to(device)
model = torch.compile(model)

optimizer = train_config.make_optimizer(model)
scheduler = train_config.make_scheduler(optimizer)
scaler = train_config.make_scaler()

get_lr = train_config.get_lr
torch.set_float32_matmul_precision('high')
save_path = f'checkpoint_step{train_config.num_steps_train}_{datetime.now():%Y%m%d_%H%M%S}.pt'


In [ ]:
#Only run this next line once between all models to download and preprocess the dataset. Comment it out afterwards


# CombinedBinDataLoader.fetch_dataset(str(PROJECT_ROOT) + '/Datasets/fineweb_1B.bin', 1000000000)


In [ ]:
train_loader, val_loader = CombinedBinDataLoader.create_loaders(
    str(PROJECT_ROOT) + '/Datasets/fineweb_1B.bin',
    train_config.batch_per_iter,
    model_config.block_size,
    train_config,
    seed=42
)

In [ ]:
#Actually train the model
torch.cuda.empty_cache()
history = train_loop(model,
                     optimizer,
                     scheduler,
                     scaler,
                     device,
                     train_loader,
                     val_loader,
                     train_config,
                     model_config,
                     save_path)

In [ ]:
estimate_loss(model, val_loader, device, train_config.num_steps_val)

In [ ]:
lag_config = LagBehindConfig()
lag_model = GPT2_Lag(lag_config, device)
#below is an isoparameter model to vanilla
lag_ckpt  = torch.load('checkpoint_step3623_20260501_212524.pt',
                        map_location=device)
lag_model.load_state_dict(lag_ckpt['model_state_dict'])
lag_model.to(device)
torch.compile(lag_model)
lag_model.eval()

In [ ]:
evaluate_next_token_correction(lag_model, lag_model, val_loader, device, prompt_len=256, num_prompts=2000, temperature=1.0, top_k=1, both_lag=True)

In [ ]:
evaluate_next_token_correction(lag_model, lag_model, val_loader, device, prompt_len=510, num_prompts=2000, temperature=1.0, top_k=1, both_lag=True)

In [ ]:
@torch.no_grad()
def generate_interleaved(model, prompt_tokens, max_new_tokens, device,
                         temperature=1.0, top_k=None):
    model.eval()
    skip_dist = model.config.lag_behind
    generated = list(prompt_tokens)
    target_len = len(prompt_tokens) + max_new_tokens

    def sample(logits):
        if temperature != 1.0:
            logits = logits / temperature
        if top_k is not None:
            k = min(top_k, logits.size(-1))
            threshold = torch.topk(logits, k).values[-1]
            logits[logits < threshold] = float('-inf')
        return torch.multinomial(F.softmax(logits, dim=-1), 1).item()

    def run_model(tokens):
        seq  = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)
        seq2 = seq.repeat(2, 1)
        logits_fwd, logits_lag, _ = model(seq2)
        return logits_fwd, logits_lag

    count_corr = 0
    while len(generated) < target_len:
        logits_fwd, logits_lag = run_model(generated)
        lag_pos = len(generated) - 1 - skip_dist

        next_token = sample(logits_fwd[0, -1, :].clone())
        generated.append(next_token)

        if lag_pos >= len(prompt_tokens):
            corrected = sample(logits_lag[0, -1, :].clone())
            if corrected != generated[lag_pos]:
                generated[lag_pos] = corrected
                count_corr += 1


    return generated, count_corr

In [ ]:
import tiktoken
enc = tiktoken.get_encoding('gpt2')

prompt = "One, Two, Three,"
prompt_tokens = enc.encode(prompt)

output_tokens, count_corr = generate_interleaved(
    model, prompt_tokens,
    max_new_tokens=100,
    device=device,
    temperature=0.8,
    top_k=50
)

print(enc.decode(output_tokens))
print(f"Total tokens generated: {len(output_tokens) - len(prompt_tokens)}")
print(f"Total corrections made: {count_corr}")

In [ ]:
import time

def _sample(logits, top_k=1):
    if top_k is not None:
        threshold = torch.topk(logits, min(top_k, logits.size(-1))).values[-1]
        logits[logits < threshold] = float('-inf')
    return torch.multinomial(F.softmax(logits, dim=-1), 1).item()

@torch.no_grad()
def benchmark_vanilla(model, prompt_tokens, max_new_tokens, device, top_k=1):
    model.eval()
    torch.cuda.synchronize()

    # prefill
    start = time.perf_counter()
    logits, kv_caches = model.prefill(prompt_tokens)
    torch.cuda.synchronize()
    prefill_time = time.perf_counter() - start

    generated  = list(prompt_tokens)
    next_token = _sample(logits[0, -1, :].clone(), top_k)
    generated.append(next_token)

    token_latencies = []
    for i in range(max_new_tokens - 1):
        torch.cuda.synchronize()
        t0 = time.perf_counter()

        logits, kv_caches = model.decode_one_token(
            next_token, len(generated) - 1, kv_caches
        )
        next_token = _sample(logits[0, -1, :].clone(), top_k)
        generated.append(next_token)

        torch.cuda.synchronize()
        token_latencies.append(time.perf_counter() - t0)

    return generated, prefill_time, token_latencies

@torch.no_grad()
def benchmark_lag(model, prompt_tokens, max_new_tokens, device, top_k=1):
    model.eval()
    torch.cuda.synchronize()

    # prefill
    start = time.perf_counter()
    logits, kv_caches = model.prefill(prompt_tokens)
    torch.cuda.synchronize()
    prefill_time = time.perf_counter() - start

    generated  = list(prompt_tokens)
    next_token = _sample(logits[0, -1, :].clone(), top_k)
    generated.append(next_token)

    token_latencies  = []
    fwd_latencies    = []
    lag_latencies    = []
    corrections      = 0

    for i in range(max_new_tokens - 1):
        torch.cuda.synchronize()
        t0 = time.perf_counter()

        logits, kv_caches = model.decode_one_token(
            next_token, len(generated) - 1, kv_caches
        )
        next_token = _sample(logits[0, -1, :].clone(), top_k)
        generated.append(next_token)
        torch.cuda.synchronize()
        fwd_latencies.append(time.perf_counter() - t0)

        tl = time.perf_counter()
        logits_lag  = model.lag_correct(generated)
        corrected   = _sample(logits_lag[0, -1, :].clone(), top_k)
        torch.cuda.synchronize()
        lag_latencies.append(time.perf_counter() - tl)

        lag_pos = len(generated) - 1 - model.config.lag_behind
        if lag_pos >= len(prompt_tokens) and corrected != generated[lag_pos]:
            generated[lag_pos] = corrected
            corrections += 1

        torch.cuda.synchronize()
        token_latencies.append(time.perf_counter() - t0)

    return generated, prefill_time, token_latencies, fwd_latencies, lag_latencies, corrections

In [ ]:
def _extend_cache(kv_caches, model, token, pos):
    x = torch.tensor([[token]], dtype=torch.long, device=model.device)
    pos_emb = model.transformer.wpe(
        torch.tensor([pos], dtype=torch.long, device=model.device)
    )
    h = model.transformer.wte(x) + pos_emb
    new_caches = []
    for layer_block, kv_cache in zip(model.transformer.h, kv_caches):
        h, cache = layer_block(h, future=False, kv_cache=kv_cache)
        new_caches.append(cache)
    return new_caches


def _update_cache_at(model, generated, lag_pos, kv_caches):
    """
    Recompute hidden state and K,V at lag_pos using corrected token
    and cached context from positions 0..lag_pos-1.
    Replace cache at lag_pos, leave all other positions unchanged.
    """
    token = generated[lag_pos]
    x = torch.tensor([[token]], dtype=torch.long, device=model.device)
    pos_emb = model.transformer.wpe(
        torch.tensor([lag_pos], dtype=torch.long, device=model.device)
    )
    h = model.transformer.wte(x) + pos_emb

    new_caches = []
    for layer_idx, (layer_block, kv_cache) in enumerate(
            zip(model.transformer.h, kv_caches)):

        k_cache, v_cache = kv_cache

        k_prefix = k_cache[:, :, :lag_pos, :]
        v_prefix = v_cache[:, :, :lag_pos, :]

        h, new_cache = layer_block(h, future=False,
                                    kv_cache=(k_prefix, v_prefix))

        k_new, v_new = new_cache
        k_full = torch.cat([k_new, k_cache[:, :, lag_pos+1:, :]], dim=2)
        v_full = torch.cat([v_new, v_cache[:, :, lag_pos+1:, :]], dim=2)
        new_caches.append((k_full, v_full))

    return new_caches

In [ ]:
@torch.no_grad()
def lag_update_cache_at(model, generated, lag_pos, lag_kv_caches):
    token   = generated[lag_pos]
    x       = torch.tensor([[token]], dtype=torch.long, device=model.device)
    pos_emb = model.transformer.wpe(
        torch.tensor([lag_pos], dtype=torch.long, device=model.device)
    )
    h = model.transformer.wte(x) + pos_emb

    new_caches = []
    for layer_block, kv_cache in zip(model.transformer.h, lag_kv_caches):
        k_cache, v_cache = kv_cache
        k_prefix = k_cache[:, :, :lag_pos, :]
        v_prefix = v_cache[:, :, :lag_pos, :]

        h, new_cache = layer_block(h, future=True,
                                    kv_cache=(k_prefix, v_prefix))

        k_new, v_new = new_cache
        k_full = torch.cat([k_new, k_cache[:, :, lag_pos+1:, :]], dim=2)
        v_full = torch.cat([v_new, v_cache[:, :, lag_pos+1:, :]], dim=2)
        new_caches.append((k_full, v_full))

    return new_caches

In [ ]:
@torch.no_grad()
def benchmark_lag_kv(model, prompt_tokens, max_new_tokens, device,
                      top_k=1, use_kv_lag=False, stale=True):
    model.eval()
    torch.cuda.synchronize()

    # prefill
    start = time.perf_counter()
    logits, kv_caches = model.prefill(prompt_tokens)
    torch.cuda.synchronize()
    prefill_time = time.perf_counter() - start


    next_token = _sample(logits[0, -1, :].clone(), top_k)

    generated = list(prompt_tokens)
    generated.append(next_token)
    kv_caches = _extend_cache(kv_caches, model, next_token,
                                    len(generated)-1)

    token_latencies = []
    fwd_latencies = []
    lag_latencies = []
    corrections = 0

    for i in range(max_new_tokens - 1):
        torch.cuda.synchronize()
        t0 = time.perf_counter()

        tf = time.perf_counter()
        logits, kv_caches = model.decode_one_token(
            next_token, len(generated)-1, kv_caches
        )
        next_token = _sample(logits[0, -1, :].clone(), top_k)
        generated.append(next_token)
        torch.cuda.synchronize()
        fwd_latencies.append(time.perf_counter() - tf)

        tl = time.perf_counter()
        lag_pos = len(generated) - 1 - model.config.lag_behind

        if lag_pos >= len(prompt_tokens):
            logits_lag = model.lag_correct(generated)
            corrected  = _sample(logits_lag[0, -1, :].clone(), top_k)

            if corrected != generated[lag_pos]:
                generated[lag_pos] = corrected
                corrections += 1

                if use_kv_lag:
                    kv_caches = _update_cache_at(
                        model, generated, lag_pos, kv_caches
                    )

                    if not stale:
                        for regen_pos in range(lag_pos + 1, len(generated)):
                            logits, kv_caches = model.decode_one_token(
                                generated[regen_pos - 1],
                                regen_pos - 1,
                                kv_caches
                            )
                            new_token = _sample(logits[0, -1, :].clone(), top_k)
                            generated[regen_pos] = new_token
                            next_token = generated[-1]

        torch.cuda.synchronize()
        lag_latencies.append(time.perf_counter() - tl)
        token_latencies.append(time.perf_counter() - t0)

    return generated, prefill_time, token_latencies, fwd_latencies, lag_latencies, corrections

In [ ]:
@torch.no_grad()
def benchmark_lag_full_kv(model, prompt_tokens, max_new_tokens, device, top_k=1):
    model.eval()
    torch.cuda.synchronize()

    start = time.perf_counter()
    fwd_logits, fwd_kv = model.prefill(prompt_tokens)
    lag_logits, lag_kv = model.lag_prefill(prompt_tokens)
    torch.cuda.synchronize()
    prefill_time = time.perf_counter() - start

    generated  = list(prompt_tokens)
    next_token = _sample(fwd_logits[0, -1, :].clone(), top_k)
    generated.append(next_token)

    fwd_logits, fwd_kv = model.decode_one_token(
        next_token, len(generated)-1, fwd_kv)
    lag_logits, lag_kv = model.lag_decode_one_token(
        next_token, len(generated)-1, lag_kv)

    token_latencies = []
    corrections = 0

    for _ in range(max_new_tokens - 1):
        torch.cuda.synchronize()
        t0 = time.perf_counter()

        fwd_logits, fwd_kv = model.decode_one_token(
            next_token, len(generated)-1, fwd_kv)
        next_token = _sample(fwd_logits[0, -1, :].clone(), top_k)
        generated.append(next_token)

        lag_logits, lag_kv = model.lag_decode_one_token(
            next_token, len(generated)-1, lag_kv)
        corrected = _sample(lag_logits[0, -1, :].clone(), top_k)

        lag_pos = len(generated) - 1 - model.config.lag_behind
        if lag_pos >= len(prompt_tokens) and corrected != generated[lag_pos]:
            generated[lag_pos] = corrected
            corrections       += 1
            fwd_kv = _update_cache_at(model, generated, lag_pos, fwd_kv)
            lag_kv = lag_update_cache_at(model, generated, lag_pos, lag_kv)

        torch.cuda.synchronize()
        token_latencies.append(time.perf_counter() - t0)

    return generated, prefill_time, token_latencies, corrections

In [ ]:
val_loader.idx = 0
x, _ = val_loader.get_data()
prompt_tokens = x[0, :50].tolist()
MAX_NEW = 1024

# config 1: no KV for lag (current baseline)
out1, pre1, lat1, fwd1, lag1, corr1 = benchmark_lag_kv(
    lag_model, prompt_tokens, MAX_NEW, device,
    use_kv_lag=False, stale=True
)

# config 2: KV + stale
out2, pre2, lat2, fwd2, lag2, corr2 = benchmark_lag_kv(
    lag_model, prompt_tokens, MAX_NEW, device,
    use_kv_lag=True, stale=True
)

# config 3: KV + regenerate (no stale)
out3, pre3, lat3, fwd3, lag3, corr3 = benchmark_lag_kv(
    lag_model, prompt_tokens, MAX_NEW, device,
    use_kv_lag=True, stale=False
)

print(f"\n{'Metric':<35} | {'No KV':>10} | {'KV+Stale':>10} | {'KV+Regen':>10}")
print("-" * 72)
print(f"{'Avg token latency (ms)':<35} | "
      f"{np.mean(lat1)*1000:>10.2f} | "
      f"{np.mean(lat2)*1000:>10.2f} | "
      f"{np.mean(lat3)*1000:>10.2f}")
print(f"{'Avg lag latency (ms)':<35} | "
      f"{np.mean(lag1)*1000:>10.2f} | "
      f"{np.mean(lag2)*1000:>10.2f} | "
      f"{np.mean(lag3)*1000:>10.2f}")
print(f"{'Total decode time (s)':<35} | "
      f"{sum(lat1):>10.3f} | "
      f"{sum(lat2):>10.3f} | "
      f"{sum(lat3):>10.3f}")
print(f"{'Throughput (tok/s)':<35} | "
      f"{MAX_NEW/sum(lat1):>10.1f} | "
      f"{MAX_NEW/sum(lat2):>10.1f} | "
      f"{MAX_NEW/sum(lat3):>10.1f}")
print(f"{'Corrections made':<35} | "
      f"{corr1:>10} | "
      f"{corr2:>10} | "
      f"{corr3:>10}")

In [ ]:
val_loader.idx = 0
x, _ = val_loader.get_data()
prompt_tokens = x[0, :50].tolist()
MAX_NEW = 200

_, pre1, lat1, fwd1, lag1, corr1 = benchmark_lag(
    lag_model, prompt_tokens, MAX_NEW, device)

_, pre2, lat2, corr2 = benchmark_lag_full_kv(
    lag_model, prompt_tokens, MAX_NEW, device)

print(f"\n{'Metric':<30} | {'No lag KV':>12} | {'Full lag KV':>12}")
print("-" * 58)
print(f"{'Prefill time (ms)':<30} | {pre1*1000:>12.2f} | {pre2*1000:>12.2f}")
print(f"{'Avg token latency (ms)':<30} | {np.mean(lat1)*1000:>12.2f} | {np.mean(lat2)*1000:>12.2f}")
print(f"{'Total decode time (s)':<30} | {sum(lat1):>12.3f} | {sum(lat2):>12.3f}")
print(f"{'Throughput (tok/s)':<30} | {MAX_NEW/sum(lat1):>12.1f} | {MAX_NEW/sum(lat2):>12.1f}")
print(f"{'Corrections':<30} | {corr1:>12} | {corr2:>12}")